In [2]:
import pandas as pd

In [3]:
df = pd.read_excel(r"C:\Users\Saroka\Downloads\Asistencia Mosca Nuevo.xlsx",sheet_name='Data cena')

In [4]:
import pandas as pd
import requests

# -----------------------
# CONFIG (ajustá si hace falta)
# -----------------------
COL_FECHA = "Fecha"       # fecha de la cena
COL_PRECIO = "Precio"     # precio en ARS
COL_USD = "Precio USD"    # columna destino

API_HIST = "https://api.argentinadatos.com/v1/cotizaciones/dolares/blue"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# -----------------------
# 1) Cargar tu df de cenas (ejemplo)
# df = pd.read_excel("cenas.xlsx")  # o pd.read_csv(...)
# -----------------------

# Parsear fecha
df[COL_FECHA] = pd.to_datetime(df[COL_FECHA], dayfirst=True, errors="coerce")
if df[COL_FECHA].isna().any():
    raise ValueError("Hay fechas inválidas en tu DF (no se pudieron parsear).")

# -----------------------
# 2) Descargar histórico completo (una vez)
# -----------------------
r = requests.get(API_HIST, headers=HEADERS, timeout=20)
r.raise_for_status()
hist = r.json()  # lista de dicts

df_fx = pd.DataFrame(hist)
df_fx["fecha_api"] = pd.to_datetime(df_fx["fecha"], errors="coerce")
df_fx = df_fx.dropna(subset=["fecha_api"]).sort_values("fecha_api")

# Si vienen como strings, aseguramos numéricos
df_fx["compra"] = pd.to_numeric(df_fx["compra"], errors="coerce")
df_fx["venta"]  = pd.to_numeric(df_fx["venta"], errors="coerce")
df_fx = df_fx.dropna(subset=["compra", "venta"])

df_fx["promedio"] = (df_fx["compra"] + df_fx["venta"]) / 2

# Nos quedamos con lo necesario
df_fx = df_fx[["fecha_api", "compra", "venta", "promedio"]]

# -----------------------
# 3) Match hacia atrás (fecha de cena -> última fecha_api <= fecha_df)
# -----------------------
df_sorted = df.sort_values(COL_FECHA).copy()

df_merged = pd.merge_asof(
    df_sorted,
    df_fx.sort_values("fecha_api"),
    left_on=COL_FECHA,
    right_on="fecha_api",
    direction="backward",   # <-- clave: siempre el más cercano para atrás
    allow_exact_matches=True
)

# Si hay fechas anteriores al inicio del histórico, puede quedar NaN
# (Opcional: levantar error o dejarlas NaN)
# print(df_merged[df_merged["fecha_api"].isna()].head())

# -----------------------
# 4) Crear df_match y Precio USD
# -----------------------
df_merged["fecha_df"] = df_merged[COL_FECHA].dt.date
df_merged["fecha_api_date"] = df_merged["fecha_api"].dt.date
df_merged["dif_dias"] = (df_merged[COL_FECHA] - df_merged["fecha_api"]).dt.days

# Precio USD = Precio / promedio(compra, venta)
df_merged[COL_USD] = df_merged[COL_PRECIO] / df_merged["promedio"]

df_match = df_merged[["fecha_df", "fecha_api_date", "dif_dias"]].rename(
    columns={"fecha_api_date": "fecha_api"}
).drop_duplicates().sort_values(["dif_dias", "fecha_df"], ascending=[False, True])

# -----------------------
# 5) Devolver df final (mismo orden original)
# -----------------------
df_final = df_merged.sort_index().drop(columns=["promedio"])  # si querés ocultar promedio



In [7]:
for fila in df_final['Precio USD']:
    print(fila)

0.0009615384615384616
0.0010152284263959391
9.35960591133005
8.415841584158416
12.0
8.205128205128204
9.547738693467336
8.457711442786069
7.464114832535885
nan
8.04368932038835
8.108108108108109
8.057851239669422
nan
7.968127490039841
9.84251968503937
6.744186046511628
7.380073800738008
5.673758865248227
nan
7.665505226480836
9.12280701754386
7.2202166064981945
nan
nan
5.970149253731344
2.3166023166023164
7.72
9.56175298804781
6.882591093117409
9.31174089068826
nan
nan
7.57201646090535
nan
10.169491525423728
9.955555555555556
4.424778761061947
17.155555555555555
11.08108108108108
18.26923076923077
15.525114155251142
nan
6.224066390041494
8.19672131147541
11.203319502074688
9.34959349593496
10.62992125984252
9.922480620155039
11.846153846153847
nan
14.574898785425102
0.0
8.936170212765957
10.300429184549357
10.38961038961039
nan
8.88888888888889
17.316017316017316
16.949152542372882
13.445378151260504
10.833333333333334
6.39344262295082
6.821705426356589
32.664092664092664
13.0268199233

In [5]:
df_final.to_excel("cenas_usd.xlsx", index=False)

In [12]:
import requests

url = "https://dolarapi.com/v1/dolares/blue"
response = requests.get(url)

data = response.json()

print(data)


{'moneda': 'USD', 'casa': 'blue', 'nombre': 'Blue', 'compra': 1505, 'venta': 1525, 'fechaActualizacion': '2025-12-30T17:58:00.000Z'}
